# Pipeline de Preparação dos Dados (Consolidação do EDA)

Este notebook consolida diretamente os insights levantados na Análise Exploratória (EDA) em um pipeline limpo do Scikit-Learn:
1. **Preenchimento de valores faltantes (Imputação)**: mediana para numéricas e moda para categóricas/booleanas;
2. **Transformação e Normalização das features sinalizadas**:
   - **`log1p` + padronização** para as variáveis com cauda longa/alta assimetria (`impressions`, `clicks`, `spend`, `conversions`, `revenue`, `roas`);
   - **`RobustScaler`** para as variáveis contínuas com presença de outliers (`video_duration_s`, `avg_watch_time_s`, `cpc`);
   - **`MinMaxScaler`** para taxas já limitadas entre 0 e 1 (`ctr`, `engagement_rate`, `music_voice_ratio`);
   - **`OneHotEncoder`** para as categóricas nominais (`platform`, `category`, `objective`, `format`);
   - **`OrdinalEncoder`** para as ordinais (`target_audience_age`, `text_density`);
   - Conversão binária para booleanas (`0` e `1`);
3. **Separação em Treino e Teste**: 80% treino e 20% teste com semente fixa.

--- 
## 1. Carregamento dos Dados e Divisão Treino / Teste

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)

# 1. Leitura do dataset
df = pd.read_csv('base_de_dados/klike_challenge_dataset.csv')

# 2. Separação de features (X) e target (y), descartando campaign_id e date
X = df.drop(columns=['klike_score', 'campaign_id', 'date'])
y = df['klike_score']

# 3. Divisão Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape[0]} linhas x {X_train.shape[1]} colunas")
print(f"X_test : {X_test.shape[0]} linhas x {X_test.shape[1]} colunas")

X_train: 400 linhas x 23 colunas
X_test : 100 linhas x 23 colunas


--- 
## 2. Construção do Pipeline (Imputação de Nulos e Normalização)

In [2]:
# Mapeamento dos grupos conforme sinalizado na EDA
cols_log = ['impressions', 'clicks', 'spend', 'conversions', 'revenue', 'roas']
cols_robust = ['video_duration_s', 'avg_watch_time_s', 'cpc']
cols_taxas = ['ctr', 'engagement_rate', 'music_voice_ratio']
cols_nominais = ['platform', 'category', 'objective', 'format']
cols_ordinais = ['target_audience_age', 'text_density']
cols_binarias = ['is_retargeting', 'has_subtitle', 'has_cta', 'has_hook', 'has_face']

# Pipeline de transformações consolidado
pipeline = ColumnTransformer(
    transformers=[
        ('log', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
            ('scaler', StandardScaler())
        ]), cols_log),
        
        ('robust', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), cols_robust),
        
        ('taxas', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', MinMaxScaler())
        ]), cols_taxas),
        
        ('nominais', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
        ]), cols_nominais),
        
        ('ordinais', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ord', OrdinalEncoder(categories=[
                ['18-24', '25-34', '35-44', '45+'],
                ['low', 'medium', 'high']
            ], handle_unknown='use_encoded_value', unknown_value=-1)),
            ('scaler', MinMaxScaler())
        ]), cols_ordinais),
        
        ('binarias', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('to_int', FunctionTransformer(lambda x: x.astype(int), feature_names_out='one-to-one'))
        ]), cols_binarias)
    ]
)

pipeline

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...), ('robust', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featur

--- 
## 3. Aplicação do Pipeline (Fit no Treino e Transform no Teste)

In [3]:
# Ajuste do pipeline no treino e transformação dos dados
X_train_prep = pipeline.fit_transform(X_train)
X_test_prep = pipeline.transform(X_test)

# Nomes das colunas geradas
feature_names = pipeline.get_feature_names_out()

# Conversão para DataFrame indexado para fácil visualização
X_train_df = pd.DataFrame(X_train_prep, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_prep, columns=feature_names, index=X_test.index)

print("=== RESULTADO FINAL DA PREPARAÇÃO ===")
print(f"X_train preparado: {X_train_df.shape} | Nulos restantes: {X_train_df.isna().sum().sum()}")
print(f"X_test preparado : {X_test_df.shape}  | Nulos restantes: {X_test_df.isna().sum().sum()}")
print(f"Total de features tratadas: {len(feature_names)}")
X_train_df.head()

=== RESULTADO FINAL DA PREPARAÇÃO ===
X_train preparado: (400, 31) | Nulos restantes: 0
X_test preparado : (100, 31)  | Nulos restantes: 0
Total de features tratadas: 31


,log__impressions,log__clicks,log__spend,log__conversions,log__revenue,log__roas,robust__video_duration_s,robust__avg_watch_time_s,robust__cpc,taxas__ctr,...,nominais__objective_traffic,nominais__format_quadrado,nominais__format_vertical,ordinais__target_audience_age,ordinais__text_density,binarias__is_retargeting,binarias__has_subtitle,binarias__has_cta,binarias__has_hook,binarias__has_face
249,1.537155,1.887202,2.358936,2.010265,0.910713,-1.111430,0.461538,1.478006,3.544041,0.751015,...,0.0,0.0,1.0,0.333333,0.5,0.0,1.0,1.0,1.0,0.0
433,1.035870,0.383424,0.161570,0.572562,0.099719,-0.642325,0.230769,0.199413,-0.248705,0.135995,...,0.0,1.0,0.0,0.000000,0.5,0.0,1.0,0.0,0.0,1.0
19,0.493220,0.731357,0.118045,1.498712,1.178476,2.682517,0.538462,0.856305,-0.808290,0.535859,...,1.0,0.0,1.0,0.666667,0.0,1.0,1.0,0.0,0.0,1.0
322,-0.404916,0.224158,0.006779,0.493634,0.834030,1.626543,-0.538462,-0.551320,0.000000,0.865359,...,0.0,0.0,1.0,0.333333,0.5,1.0,0.0,1.0,1.0,1.0
332,-0.526822,-0.422915,-0.179091,-0.896216,-0.286120,-0.906229,1.846154,2.826979,0.911917,0.368742,...,1.0,0.0,0.0,0.666667,0.5,0.0,1.0,1.0,1.0,1.0
